1. Загрузка данных

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_excel('/kaggle/input/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction/E Commerce Dataset.xlsx', sheet_name='E Comm')

2. Обработка пропусков (Медиана)

In [2]:
cols_with_nan = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
for col in cols_with_nan:
    df[col] = df[col].fillna(df[col].median())

3. Подготовка данных для модели (ML-часть)

In [3]:
X = df.drop(['CustomerID', 'Churn'], axis=1)
y = df['Churn']

Кодирование категориальных признаков

In [4]:
le = LabelEncoder()
cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

4. Обучение модели

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

5. Оценка качества (Recall)

In [6]:
y_pred = model.predict(X_test)
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.2%}")

Recall: 74.05%


6. Добавление прогнозов в основной датасет

In [7]:
df['Churn_Probability'] = model.predict_proba(X)[:, 1]
df['Churn_Prediction'] = model.predict(X)

7. Экспорт основной витрины

In [8]:
df.to_csv('churn_results_for_BI.csv', index=False)

8. Экспорт Важности Признаков (Feature Importance)

In [9]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [10]:
importance.to_csv('feature_importance.csv', index=False)

Файлы churn_results_for_BI.csv и feature_importance.csv готовы к импорту в Power BI!